# DevRev Search Dataset

Loading and exploring the `devrev/search` dataset from Hugging Face.

**Dataset Structure:**
- `annotated_queries` — Queries paired with annotated (golden) article chunks
- `knowledge_base` — Article chunks from DevRev's customer-facing support documentation
- `test_queries` — Held-out queries used for evaluation

In [ ]:
from datasets import load_dataset
import pandas as pd

## 1. Load Annotated Queries
Queries paired with annotated (golden) article chunks for training/validation.

In [ ]:
# Load annotated queries
annotated_queries = load_dataset("devrev/search", "annotated_queries")
print(annotated_queries)

In [ ]:
# Convert to DataFrame and display
annotated_df = annotated_queries["train"].to_pandas()
annotated_df.head()

In [ ]:
# Sample a single annotated query example
annotated_queries["train"][0]

## 2. Load Test Queries
Held-out queries used for evaluation.

In [ ]:
# Load test queries
test_queries = load_dataset("devrev/search", "test_queries")
print(test_queries)

In [ ]:
# Convert to DataFrame and display
test_df = test_queries["test"].to_pandas()
test_df.head()

In [ ]:
# Sample a single test query example
test_queries["test"][0]

## 3. Load Knowledge Base
Article chunks from DevRev's customer-facing support documentation.

In [ ]:
# Load knowledge base
knowledge_base = load_dataset("devrev/search", "knowledge_base")
print(knowledge_base)

In [ ]:
# Convert to DataFrame and display
knowledge_df = knowledge_base["corpus"].to_pandas()
knowledge_df.head()

In [ ]:
# Sample a single knowledge base chunk
knowledge_base["corpus"][0]

## 4. Dataset Summary

In [ ]:
print("=" * 60)
print("DevRev Search Dataset Summary")
print("=" * 60)
print(f"\nAnnotated Queries:")
print(annotated_queries)
print(f"\nTest Queries:")
print(test_queries)
print(f"\nKnowledge Base:")
print(knowledge_base)
print("\n" + "=" * 60)

---
## 5. Index Knowledge Base with FAISS

Using OpenAI text-embedding-3-small and FAISS for similarity search.

In [ ]:
from openai import OpenAI
import faiss
import numpy as np
from tqdm import tqdm
import time
import os
import json
import pickle

In [ ]:
# Initialize OpenAI client
# Set your API key as an environment variable: export OPENAI_API_KEY="your-key-here"
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Please set OPENAI_API_KEY environment variable")

client = OpenAI(api_key=OPENAI_API_KEY)

MODEL_ID = "text-embedding-3-small"  # 1536 dimensions
print(f"Using model: {MODEL_ID}")
print(f"Provider: OpenAI")

In [ ]:
def get_embedding(text: str) -> np.ndarray:
    """Get embedding for a single text using OpenAI text-embedding-3."""
    response = client.embeddings.create(
        model=MODEL_ID,
        input=text
    )
    return np.array(response.data[0].embedding)

In [ ]:
def get_embeddings_batch(texts: list) -> np.ndarray:
    """Get embeddings for a batch of texts using OpenAI text-embedding-3."""
    response = client.embeddings.create(
        model=MODEL_ID,
        input=texts
    )
    return np.array([d.embedding for d in response.data])

In [ ]:
# Test the embedding API
test_embedding = get_embedding("This is a test sentence.")
print(f"Test embedding shape: {test_embedding.shape}")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

In [ ]:
# Prepare documents: concatenate title with text
corpus = knowledge_base["corpus"]

documents = []
doc_ids = []
doc_titles = []
doc_texts = []

for item in tqdm(corpus, desc="Preparing documents"):
    # Concatenate title and text
    doc_text = f"{item['title']}\n\n{item['text']}"
    documents.append(doc_text)
    doc_ids.append(item['id'])
    doc_titles.append(item['title'])
    doc_texts.append(item['text'])

print(f"\nTotal documents: {len(documents):,}")
print(f"\nSample document:")
print(documents[0][:500])

In [ ]:
def get_all_embeddings(texts, batch_size=100, max_retries=3):
    """Get embeddings for all texts using OpenAI batch API."""
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch = texts[i:i + batch_size]
        
        # Truncate texts if too long (OpenAI has ~8K token limit)
        batch = [text[:8000] if len(text) > 8000 else text for text in batch]
        
        retries = 0
        while retries < max_retries:
            try:
                embeddings = get_embeddings_batch(batch)
                all_embeddings.append(embeddings)
                break
            except Exception as e:
                retries += 1
                if retries >= max_retries:
                    print(f"Error embedding batch {i}: {e}")
                    all_embeddings.append(np.zeros((len(batch), 1536)))
                else:
                    print(f"Retry {retries} for batch {i}...")
                    time.sleep(2)
        
        time.sleep(0.1)  # Rate limiting
    
    return np.vstack(all_embeddings)

In [ ]:
# Generate embeddings for all documents
print("Generating embeddings for knowledge base...")
print(f"Total documents: {len(documents):,}")
print("Using batch processing for efficiency...")

# For testing, use a subset (uncomment for all documents)
# documents_to_embed = documents[:100]  # Test with first 100 docs
documents_to_embed = documents  # All documents

embeddings = get_all_embeddings(documents_to_embed, batch_size=1000)
print(f"\nEmbeddings shape: {embeddings.shape}")

# Save embeddings
np.save("embeddings.npy", embeddings)
print("Embeddings saved to embeddings.npy")

In [ ]:
# Load embeddings if already saved
# embeddings = np.load("embeddings.npy")
# print(f"Loaded embeddings shape: {embeddings.shape}")

In [ ]:
# Create FAISS index
embedding_dim = embeddings.shape[1]
print(f"Creating FAISS index with dimension: {embedding_dim}")

# Normalize embeddings for cosine similarity
embeddings_normalized = embeddings.copy().astype('float32')
faiss.normalize_L2(embeddings_normalized)

# Create the index using IndexFlatIP for inner product (cosine similarity with normalized vectors)
index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings_normalized)

print(f"Index created with {index.ntotal:,} vectors")

In [ ]:
# Save the index and document mapping
INDEX_DIR = "faiss_index"
os.makedirs(INDEX_DIR, exist_ok=True)

# Save FAISS index
faiss.write_index(index, os.path.join(INDEX_DIR, "knowledge_base.index"))

# Save document mapping
with open(os.path.join(INDEX_DIR, "doc_mapping.pkl"), "wb") as f:
    pickle.dump({
        "doc_ids": doc_ids,
        "documents": documents,
        "doc_titles": doc_titles,
        "doc_texts": doc_texts
    }, f)

print(f"✓ Index saved to {INDEX_DIR}/knowledge_base.index")
print(f"✓ Document mapping saved to {INDEX_DIR}/doc_mapping.pkl")

## 6. Search the Knowledge Base

In [ ]:
def search(query: str, k: int = 5):
    """Search the knowledge base for relevant documents."""
    query_embedding = get_embedding(query).astype('float32')
    query_embedding = query_embedding.reshape(1, -1)
    faiss.normalize_L2(query_embedding)
    
    scores, indices = index.search(query_embedding, k)
    
    results = []
    for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
        results.append({
            "rank": i + 1,
            "score": float(score),
            "id": doc_ids[idx],
            "title": doc_titles[idx],
            "text": doc_texts[idx]
        })
    
    return results

In [ ]:
# Test search with a sample query
query = "How do I set up AirSync?"
results = search(query, k=5)

print(f"Query: {query}")
print("=" * 60)

for r in results:
    print(f"\n[Rank {r['rank']}] Score: {r['score']:.4f}")
    print(f"Doc ID: {r['id']}")
    print(f"Title: {r['title']}")
    print(f"Text: {r['text'][:300]}...")
    print("-" * 40)

---
## 7. Run Evaluation on Test Queries

Run search against all test queries and save results in the same format as annotated_queries.

In [ ]:
# Run search on all test queries - DIRECT VERSION (no dependency on search function)
TOP_K = 10  # Number of retrievals per query

# Get corpus data for lookups
corpus_data = knowledge_base["corpus"]

test_results = []

for item in tqdm(test_queries["test"], desc="Processing test queries"):
    query_id = item["query_id"]
    query = item["query"]
    
    # Get query embedding directly
    query_embedding = get_embedding(query).astype('float32').reshape(1, -1)
    faiss.normalize_L2(query_embedding)
    
    # Search FAISS index
    scores, indices = index.search(query_embedding, TOP_K)
    
    # Format retrievals using corpus data directly
    retrievals = []
    for idx in indices[0]:
        doc = corpus_data[int(idx)]
        retrievals.append({
            "id": doc["id"],
            "text": doc["text"],
            "title": doc["title"]
        })
    
    test_results.append({
        "query_id": query_id,
        "query": query,
        "retrievals": retrievals
    })

print(f"\nProcessed {len(test_results)} test queries")

In [ ]:
# Preview a sample result
import json
print("Sample result:")
print(json.dumps(test_results[0], indent=2, default=str)[:1500])

In [ ]:
# Save results to JSON file
OUTPUT_FILE = "test_queries_results.json"

with open(OUTPUT_FILE, "w") as f:
    json.dump(test_results, f, indent=2)

print(f"✓ Results saved to {OUTPUT_FILE}")
print(f"  - {len(test_results)} queries")
print(f"  - {TOP_K} retrievals per query")

In [ ]:
# Also save as a parquet file for easier loading
results_df = pd.DataFrame(test_results)
results_df.to_parquet("test_queries_results.parquet", index=False)
print("✓ Results also saved to test_queries_results.parquet")

In [ ]:
# Display results summary
print("=" * 60)
print("Test Queries Results Summary")
print("=" * 60)
print(f"Total queries: {len(test_results)}")
print(f"Retrievals per query: {TOP_K}")
print(f"\nOutput files:")
print(f"  - test_queries_results.json")
print(f"  - test_queries_results.parquet")
print("\nFormat matches annotated_queries structure:")
print("  - query_id: string")
print("  - query: string")
print("  - retrievals: list of {id, text, title}")

## 8. Load Saved Index (Optional)
Use this to load a previously saved index without re-embedding.

In [ ]:
# Load saved index and mapping
# INDEX_DIR = "faiss_index"
# index = faiss.read_index(os.path.join(INDEX_DIR, "knowledge_base.index"))
# with open(os.path.join(INDEX_DIR, "doc_mapping.pkl"), "rb") as f:
#     mapping = pickle.load(f)
#     doc_ids = mapping["doc_ids"]
#     documents = mapping["documents"]
#     doc_titles = mapping["doc_titles"]
#     doc_texts = mapping["doc_texts"]
# print(f"Loaded index with {index.ntotal:,} vectors")